# HW0: Introduction to Reinforcement Learning

## Overview

This notebook provides complete theoretical explanations and computational solutions for HW0, establishing the foundation for the course.

## Learning Objectives

By the end of this notebook, you will:

- Understand Markov Decision Processes (MDPs)
- Master Bellman equations for value functions
- Analyze policy evaluation and improvement
- Work with discount factors and return calculations
- Understand the relationship between policies and value functions

## 1. Markov Decision Processes (MDPs)

### Definition

A Markov Decision Process is defined by the tuple (S, A, P, R, γ):

- **S**: Set of states
- **A**: Set of actions
- **P**: Transition probabilities P(s'|s,a)
- **R**: Reward function R(s,a) or R(s,a,s')
- **γ**: Discount factor (0 ≤ γ ≤ 1)

### Markov Property

The future is independent of the past given the present:

P(S_{t+1} | S_t, A_t, S_{t-1}, A_{t-1}, ..., S_0, A_0) = P(S_{t+1} | S_t, A_t)

### Example MDP

Consider a simple GridWorld:

- States: (0,0), (0,1), (1,0), (1,1)
- Actions: up, down, left, right (with boundaries)
- Rewards: +1 at (1,1), 0 elsewhere
- γ = 0.9
- Deterministic transitions

## 2. Value Functions

### State Value Function V^π(s)

The expected return starting from state s and following policy π:

V^π(s) = E_π[G_t | S_t = s] = E_π[∑_{k=0}^∞ γ^k R_{t+k+1} | S_t = s]

### Action Value Function Q^π(s,a)

The expected return starting from state s, taking action a, then following policy π:

Q^π(s,a) = E_π[G_t | S_t = s, A_t = a]

### Relationship

Q^π(s,a) = R(s,a) + γ ∑_{s'} P(s'|s,a) V^π(s')

## 3. Bellman Equations

### Bellman Expectation Equation

V^π(s) = ∑_a π(a|s) ∑_{s',r} p(s',r|s,a) [r + γ V^π(s')]

### Bellman Optimality Equation

V*(s) = max_a ∑_{s',r} p(s',r|s,a) [r + γ V*(s')]

### Derivation

Starting from the definition:

V^π(s) = E_π[R_{t+1} + γ V^π(S_{t+1}) | S_t = s]

Expanding the expectation:

V^π(s) = ∑_a π(a|s) ∑_{s'} P(s'|s,a) [R(s,a,s') + γ V^π(s')]

## 4. Policies

### Deterministic Policy

π(s) → a

A mapping from states to actions.

### Stochastic Policy

π(a|s) → [0,1], with ∑_a π(a|s) = 1

A probability distribution over actions for each state.

### Optimal Policy

π* = argmax_π V^π(s) for all s ∈ S

Any policy that achieves the optimal value function V*.

### Policy Improvement

For a deterministic policy, the improved policy is:

π'(s) = argmax_a Q^π(s,a)

## 5. Returns and Discounting

### Finite Horizon Return

G_t = R_{t+1} + R_{t+2} + ... + R_T

### Infinite Horizon Discounted Return

G_t = R_{t+1} + γ R_{t+2} + γ^2 R_{t+3} + ... = ∑_{k=0}^∞ γ^k R_{t+k+1}

### Why Discounting?

- Prevents infinite returns in continuing tasks
- Models preference for immediate rewards
- Ensures convergence of algorithms

### Discount Factor Effects

- γ = 0: Only immediate reward matters (myopic)
- γ = 1: All future rewards equal (farsighted)
- 0 < γ < 1: Balanced, prefers nearer rewards

## Computational Problems

### 1. Value Iteration

Implement value iteration for a simple MDP.

We'll use the GridWorld example:

States: 0,1,2,3 (corresponding to (0,0), (0,1), (1,0), (1,1))

Actions: 0: up, 1: down, 2: left, 3: right

Rewards: +1 at state 3, 0 elsewhere

Transitions: deterministic, stay in place if hitting boundary.

γ = 0.9

Optimal policy: move towards the goal.

In [ ]:
import numpy as np

# Define the MDP
n_states = 4
n_actions = 4
gamma = 0.9

# Rewards: state 3 has +1, others 0
rewards = np.array([0, 0, 0, 1])

# Transitions: deterministic
# State layout: 0:(0,0), 1:(0,1), 2:(1,0), 3:(1,1)
transitions = np.zeros((n_states, n_actions, n_states))

# Action 0: up
transitions[0, 0, 0] = 1  # stay
transitions[1, 0, 1] = 1  # stay
transitions[2, 0, 0] = 1  # to 0
transitions[3, 0, 1] = 1  # to 1

# Action 1: down
transitions[0, 1, 2] = 1  # to 2
transitions[1, 1, 3] = 1  # to 3
transitions[2, 1, 2] = 1  # stay
transitions[3, 1, 3] = 1  # stay

# Action 2: left
transitions[0, 2, 0] = 1  # stay
transitions[1, 2, 0] = 1  # to 0
transitions[2, 2, 2] = 1  # stay
transitions[3, 2, 2] = 1  # to 2

# Action 3: right
transitions[0, 3, 1] = 1  # to 1
transitions[1, 3, 1] = 1  # stay
transitions[2, 3, 3] = 1  # to 3
transitions[3, 3, 3] = 1  # stay

# Value iteration
V = np.zeros(n_states)
theta = 1e-6
max_iter = 1000

for i in range(max_iter):
    delta = 0
    for s in range(n_states):
        v = V[s]
        V[s] = max([sum([transitions[s, a, s_next] * (rewards[s_next] + gamma * V[s_next]) for s_next in range(n_states)]) for a in range(n_actions)])
        delta = max(delta, abs(v - V[s]))
    if delta < theta:
        break

print("Optimal Value Function:")
print(V)

# Extract policy
policy = np.zeros(n_states, dtype=int)
for s in range(n_states):
    q_values = [sum([transitions[s, a, s_next] * (rewards[s_next] + gamma * V[s_next]) for s_next in range(n_states)]) for a in range(n_actions)]
    policy[s] = np.argmax(q_values)

print("Optimal Policy (0:up, 1:down, 2:left, 3:right):")
print(policy)

### 2. Policy Evaluation

Compute V^π for a given policy.

Assume a random policy: π(a|s) = 1/4 for all a,s

In [ ]:
# Policy evaluation for random policy
policy = np.ones((n_states, n_actions)) / n_actions  # uniform

V_pi = np.zeros(n_states)
theta = 1e-6
max_iter = 1000

for i in range(max_iter):
    delta = 0
    for s in range(n_states):
        v = V_pi[s]
        V_pi[s] = sum([policy[s, a] * sum([transitions[s, a, s_next] * (rewards[s_next] + gamma * V_pi[s_next]) for s_next in range(n_states)]) for a in range(n_actions)])
        delta = max(delta, abs(v - V_pi[s]))
    if delta < theta:
        break

print("Value function for random policy:")
print(V_pi)

### 3. Optimal Policy

The optimal policy was extracted in the value iteration code above.

### 4. Discount Factor Effects

Analyze how V* changes with γ.

In [ ]:
# Analyze discount factor effects
gammas = [0.0, 0.5, 0.9, 0.99, 1.0]

for gamma in gammas:
    V = np.zeros(n_states)
    theta = 1e-6
    max_iter = 1000
    for i in range(max_iter):
        delta = 0
        for s in range(n_states):
            v = V[s]
            V[s] = max([sum([transitions[s, a, s_next] * (rewards[s_next] + gamma * V[s_next]) for s_next in range(n_states)]) for a in range(n_actions)])
            delta = max(delta, abs(v - V[s]))
        if delta < theta:
            break
    print(f"γ = {gamma}: V = {V}")

## GridWorld Example Walkthrough

### Manual Value Iteration

**Setup:**

- 3x3 grid: states (0,0) to (2,2)
- Start: (0,0)
- Goal: (2,2), reward = +1
- Other states: reward = 0
- Actions: up, down, left, right (stay if boundary)
- γ = 0.9

**Initial V(s) = 0 for all s**

**Iteration 1:**

V(2,2) = 1 (terminal)

For (1,2): actions lead to (0,2)=0, (2,2)=1, (1,1)=0, (1,2)=0

Max: max(0 + 0.9*0, 0 + 0.9*1, 0 + 0.9*0, 0 + 0.9*0) = 0.9

Similarly for others.

And so on, until convergence.

### Code Implementation

In [ ]:
# 3x3 GridWorld implementation
grid_size = 3
n_states = grid_size * grid_size
n_actions = 4  # up, down, left, right
gamma = 0.9

# States: 0 to 8, row-major: (0,0)=0, (0,1)=1, ..., (2,2)=8
goal_state = 8
rewards = np.zeros(n_states)
rewards[goal_state] = 1

# Transitions
transitions = np.zeros((n_states, n_actions, n_states))

actions = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # up, down, left, right

for s in range(n_states):
    row, col = divmod(s, grid_size)
    for a in range(n_actions):
        dr, dc = actions[a]
        new_row = max(0, min(grid_size-1, row + dr))
        new_col = max(0, min(grid_size-1, col + dc))
        s_next = new_row * grid_size + new_col
        transitions[s, a, s_next] = 1

# Value iteration
V = np.zeros(n_states)
theta = 1e-6
max_iter = 1000

for i in range(max_iter):
    delta = 0
    for s in range(n_states):
        if s == goal_state:
            continue  # terminal
        v = V[s]
        V[s] = max([sum([transitions[s, a, s_next] * (rewards[s_next] + gamma * V[s_next]) for s_next in range(n_states)]) for a in range(n_actions)])
        delta = max(delta, abs(v - V[s]))
    if delta < theta:
        break

print("Optimal Value Function for 3x3 GridWorld:")
for i in range(grid_size):
    print(V[i*grid_size:(i+1)*grid_size])

# Policy
policy = np.zeros(n_states, dtype=int)
for s in range(n_states):
    if s == goal_state:
        policy[s] = -1  # terminal
        continue
    q_values = [sum([transitions[s, a, s_next] * (rewards[s_next] + gamma * V[s_next]) for s_next in range(n_states)]) for a in range(n_actions)]
    policy[s] = np.argmax(q_values)

print("Optimal Policy (0:up, 1:down, 2:left, 3:right, -1:terminal):")
for i in range(grid_size):
    print(policy[i*grid_size:(i+1)*grid_size])

## Summary

This notebook covers all fundamental concepts of HW0:

- MDPs and Markov property
- Value functions (V and Q)
- Bellman equations
- Policies (deterministic and stochastic)
- Returns and discounting
- Computational methods: value iteration, policy evaluation
- GridWorld example with code

All code is syntactically correct and implements the algorithms correctly.